# Google Colab Official Training Notebook for Dilated-SE-FireNet

Notebook này được thiết kế để chạy trên **Google Colab Web** sử dụng **GPU (Tesla T4)**.
Nó sử dụng khóa tài khoản dịch vụ **Google Service Account (`dilated-se-fire-10963f0a6a5a.json`)** để tự động tải dữ liệu và tải lên mô hình kết quả mà không cần mount tài khoản Google Drive cá nhân của bạn.

### Hướng dẫn chuẩn bị chạy trên Google Colab Web:
1. Mở [Google Colab Web](https://colab.research.google.com/), chọn **Upload** và tải tệp `train_colab.ipynb` này lên.
2. Trong giao diện Colab, nhìn sang thanh bên trái, click vào biểu tượng **Thư mục (Files)**, sau đó bấm nút **Upload to session storage** để tải tệp khóa **`dilated-se-fire-10963f0a6a5a.json`** (tệp key service account) lên máy ảo Colab.
3. Chọn GPU: Vào menu **Runtime -> Change runtime type -> Chọn T4 GPU**.
4. Bấm **Runtime -> Run all** để bắt đầu chạy tự động.

### 1. Cài đặt thư viện và Tải dữ liệu tự động qua Service Account

In [ ]:
# Cài đặt thư viện bổ sung
!pip install -q wfdb google-api-python-client google-auth

import os
import io
import zipfile
from pathlib import Path
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

KEY_FILE = 'dilated-se-fire-10963f0a6a5a.json'
FILE_ID = '1wQzaLcd-MSX1WnyZOSOG4JFchzq2N8SN'  # ID của file processed.zip trên Drive
OUT_FILE = 'processed.zip'

if os.path.exists(KEY_FILE):
    print('Đang xác thực Service Account và tải dữ liệu từ Google Drive...')
    try:
        creds = service_account.Credentials.from_service_account_file(
            KEY_FILE,
            scopes=['https://www.googleapis.com/auth/drive']
        )
        drive_service = build('drive', 'v3', credentials=creds)
        request = drive_service.files().get_media(fileId=FILE_ID)

        # Tải file nén dung lượng lớn theo khối (chunk) 100MB
        with open(OUT_FILE, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request, chunksize=100*1024*1024)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f'Tốc độ tải dữ liệu: {int(status.progress() * 100)}%')

        print('Tải file processed.zip thành công! Đang giải nén...')
        with zipfile.ZipFile(OUT_FILE, 'r') as zip_ref:
            zip_ref.extractall('.')
        print('Giải nén hoàn tất! Dữ liệu đã sẵn sàng.')
    except Exception as e:
        print(f'Lỗi tải hoặc giải nén dữ liệu: {e}')
else:
    print(f'LỖI: Chưa tìm thấy tệp key {KEY_FILE}!')
    print('Vui lòng click biểu tượng thư mục ở thanh bên trái của Colab và upload tệp key lên trước khi chạy!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 90.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
Đang xác thực Service Account và tải dữ liệu từ Google Drive...
Tốc độ tải dữ liệu: 2%
Tốc độ tải dữ liệu: 5%
Tố

### 2. Cấu hình thông số (Self-contained Config)

In [ ]:
from pathlib import Path

# Thiết lập các đường dẫn cục bộ trên Colab
DATA_DIR = Path('./database/processed')
OUTPUT_DIR = Path('./outputs')
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
REPORT_DIR = OUTPUT_DIR / 'reports'

for p in [CKPT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Siêu tham số ECG
TARGET_FS = 250
WINDOW_SECONDS = 10
WINDOW_SIZE = TARGET_FS * WINDOW_SECONDS  # 2500
INPUT_SHAPE = (WINDOW_SIZE, 1)

LABEL_NORMAL = 0
LABEL_AFIB = 1
NUM_CLASSES = 2

# Tham số mô hình
L2_RATE = 1e-3
LEAKY_RELU_ALPHA = 0.1
SE_RATIO = 4
DROPOUT_CLASSIFIER = 0.2
SPATIAL_DROPOUT = 0.2
STEM_FILTERS = 12

FIRE_BLOCKS = [
    ('block1', 12, 24, 1, 48),
    ('block2', 16, 32, 2, 64),
    ('block3', 24, 48, 4, 96),
    ('block4', 32, 64, 8, 128),
]

# Tham số huấn luyện
SEED = 42
BATCH_SIZE = 256
EPOCHS = 60
INITIAL_LR = 1e-3
LABEL_SMOOTHING = 0.1
PATIENCE_EARLY_STOP = 10

print('Cấu hình tham số huấn luyện hoàn tất.')

Cấu hình tham số huấn luyện hoàn tất.


### 3. Định nghĩa Hàm tiện ích và Load dữ liệu

In [ ]:
import random
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def ensure_input_shape(X):
    if X.ndim == 2:
        X = X[..., np.newaxis]
    return X

def load_processed_data(data_dir=DATA_DIR):
    X_train = np.load(data_dir / 'train' / 'X_train.npy').astype('float32')
    y_train = np.load(data_dir / 'train' / 'y_train.npy').astype('int32')
    X_val = np.load(data_dir / 'val' / 'X_val.npy').astype('float32')
    y_val = np.load(data_dir / 'val' / 'y_val.npy').astype('int32')

    X_train = ensure_input_shape(X_train)
    X_val = ensure_input_shape(X_val)

    print('Loaded Dataset:')
    print('  X_train:', X_train.shape, 'y_train:', y_train.shape)
    print('  X_val:  ', X_val.shape, 'y_val:  ', y_val.shape)
    return X_train, y_train, X_val, y_val

def to_one_hot(y):
    return tf.keras.utils.to_categorical(y, num_classes=NUM_CLASSES)

def get_class_weights(y_train):
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    return {int(c): float(w) for c, w in zip(classes, weights)}

### 4. Định nghĩa Kiến trúc mô hình Dilated-SE-FireNet

In [ ]:
from tensorflow.keras import layers, models, regularizers

def conv_bn_lrelu(x, filters, kernel_size, strides=1, dilation_rate=1, name=None):
    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=strides,
        dilation_rate=dilation_rate,
        padding='same',
        use_bias=False,
        kernel_regularizer=regularizers.l2(L2_RATE),
        name=name,
    )(x)
    x = layers.BatchNormalization(name=None if name is None else f'{name}_bn')(x)
    x = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA, name=None if name is None else f'{name}_lrelu')(x)
    return x

def se_block(x, ratio=SE_RATIO, name='se'):
    channels = int(x.shape[-1])
    hidden = max(channels // ratio, 1)

    se = layers.GlobalAveragePooling1D(name=f'{name}_gap')(x)
    se = layers.Dense(
        hidden,
        use_bias=False,
        kernel_regularizer=regularizers.l2(L2_RATE),
        name=f'{name}_fc1',
    )(se)
    se = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA, name=f'{name}_lrelu')(se)
    se = layers.Dense(
        channels,
        activation='sigmoid',
        use_bias=False,
        kernel_regularizer=regularizers.l2(L2_RATE),
        name=f'{name}_fc2_sigmoid',
    )(se)
    se = layers.Reshape((1, channels), name=f'{name}_reshape')(se)
    return layers.Multiply(name=f'{name}_scale')([x, se])

def dilated_fire_module(x, squeeze_filters, expand_filters, dilation_rate, block_name):
    shortcut = x
    s = conv_bn_lrelu(x, filters=squeeze_filters, kernel_size=1, strides=1, name=f'fire_{block_name}_squeeze_1x1')

    e1 = layers.Conv1D(
        filters=expand_filters, kernel_size=1, padding='same', use_bias=False,
        kernel_regularizer=regularizers.l2(L2_RATE), name=f'fire_{block_name}_expand_1x1'
    )(s)
    e1 = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA, name=f'fire_{block_name}_expand_1x1_lrelu')(e1)

    e3 = layers.Conv1D(
        filters=expand_filters, kernel_size=3, dilation_rate=dilation_rate, padding='same', use_bias=False,
        kernel_regularizer=regularizers.l2(L2_RATE), name=f'fire_{block_name}_expand_3x3_dil{dilation_rate}'
    )(s)
    e3 = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA, name=f'fire_{block_name}_expand_3x3_lrelu')(e3)

    out = layers.Concatenate(axis=-1, name=f'fire_{block_name}_concat')([e1, e3])
    out = layers.BatchNormalization(name=f'fire_{block_name}_concat_bn')(out)
    out = se_block(out, ratio=SE_RATIO, name=f'fire_{block_name}_se')

    out_channels = int(out.shape[-1])
    shortcut_channels = int(shortcut.shape[-1])

    if shortcut_channels != out_channels:
        shortcut = layers.Conv1D(
            filters=out_channels, kernel_size=1, padding='same', use_bias=False,
            kernel_regularizer=regularizers.l2(L2_RATE), name=f'fire_{block_name}_shortcut_proj'
        )(shortcut)
        shortcut = layers.BatchNormalization(name=f'fire_{block_name}_shortcut_bn')(shortcut)

    out = layers.Add(name=f'fire_{block_name}_add')([out, shortcut])
    out = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA, name=f'fire_{block_name}_out_lrelu')(out)
    out = layers.SpatialDropout1D(SPATIAL_DROPOUT, name=f'fire_{block_name}_spatial_dropout')(out)
    return out

def learnable_downsample(x, filters, block_name):
    return conv_bn_lrelu(x, filters=filters, kernel_size=3, strides=2, name=f'{block_name}_learnable_downsample')

def build_dilated_se_firenet(input_shape=INPUT_SHAPE):
    inputs = layers.Input(shape=input_shape, name='Input_ECG')
    x = inputs

    x = conv_bn_lrelu(x, filters=STEM_FILTERS, kernel_size=7, strides=2, name='Stem_Conv1D_k7_s2')

    for i, (name, squeeze, expand, dilation, out_channels) in enumerate(FIRE_BLOCKS):
        x = dilated_fire_module(x, squeeze_filters=squeeze, expand_filters=expand, dilation_rate=dilation, block_name=name)
        if i < len(FIRE_BLOCKS) - 1:
            x = learnable_downsample(x, filters=out_channels, block_name=f'down_after_{name}')

    x = layers.GlobalAveragePooling1D(name='Global_Average_Pooling')(x)
    x = layers.Dropout(DROPOUT_CLASSIFIER, name='Classifier_Dropout')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='Output_Softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs, name='Dilated_SE_FireNet')

### 6. Pipeline Huấn luyện

In [ ]:
def augment_ecg(x, y):
    # CPU Data Augmentation trong tf.data
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.03, dtype=tf.float32)
    amp = tf.random.uniform([], minval=0.9, maxval=1.1, dtype=tf.float32)
    offset = tf.random.uniform([], minval=-0.05, maxval=0.05, dtype=tf.float32)
    x_aug = x * amp + noise + offset
    return x_aug, y

# 1. Setup seed
set_seed()

# 2. Load data
X_train, y_train, X_val, y_val = load_processed_data()
y_train_oh = to_one_hot(y_train)
y_val_oh = to_one_hot(y_val)

# 3. Build tf.data pipeline
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))
    .shuffle(buffer_size=10000, seed=SEED, reshuffle_each_iteration=True)
    .map(augment_ecg, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val_oh))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# 4. Build Model
model = build_dilated_se_firenet()
model.summary()

# 5. Cosine Decay Optimizer
steps_per_epoch = max(len(X_train) // BATCH_SIZE, 1)
decay_steps = steps_per_epoch * EPOCHS
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=INITIAL_LR, decay_steps=decay_steps, alpha=0.05
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# 6. Compile
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision'),
    ],
)

class_weight = get_class_weights(y_train)
print('Class weights:', class_weight)

# 7. Callbacks
ckpt_path = CKPT_DIR / 'best_dilated_se_firenet.keras'
cb = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ckpt_path), monitor='val_loss', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE_EARLY_STOP, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(REPORT_DIR / 'train_log.csv')),
]

# 8. Train
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, class_weight=class_weight)

# 9. Save final weights
final_path = CKPT_DIR / 'final_dilated_se_firenet.keras'
model.save(final_path)
print('Huấn luyện thành công!')

Loaded Dataset:
  X_train: (250734, 2500, 1) y_train: (250734,)
  X_val:   (43161, 2500, 1) y_val:   (43161,)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "Dilated_SE_FireNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_ECG           │ (None, 2500, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Stem_Conv1D_k7_s2   │ (None, 1250, 12)  │         84 │ Input_ECG[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Stem_Conv1D_k7_s2_… │ (None, 1250, 12)  │         48 │ Stem_Conv1D_k7_s… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Stem_Conv1D_k7_s2_… │ (None, 1250, 12)  │          0 │ Stem_Conv1D_k7_s… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_squeez… │ (None, 1250, 12)  │        144 │ Stem_Conv1D_k7_s… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_squeez… │ (None, 1250, 12)  │         48 │ fire_block1_sque… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_squeez… │ (None, 1250, 12)  │          0 │ fire_block1_sque… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_expand… │ (None, 1250, 24)  │        288 │ fire_block1_sque… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_expand… │ (None, 1250, 24)  │        864 │ fire_block1_sque… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_expand… │ (None, 1250, 24)  │          0 │ fire_block1_expa… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_expand… │ (None, 1250, 24)  │          0 │ fire_block1_expa… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_concat  │ (None, 1250, 48)  │          0 │ fire_block1_expa… │
│ (Concatenate)       │                   │            │ fire_block1_expa… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_concat… │ (None, 1250, 48)  │        192 │ fire_block1_conc… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_se_gap  │ (None, 48)        │          0 │ fire_block1_conc… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_se_fc1  │ (None, 12)        │        576 │ fire_block1_se_g… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_se_lre… │ (None, 12)        │          0 │ fire_block1_se_f… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fire_block1_se_fc2… │ (None, 48)        │        576 │ fire_block1_se_l

 Total params: 110,694 (432.40 KB)

 Trainable params: 108,742 (424.77 KB)

 Non-trainable params: 1,952 (7.62 KB)

Class weights: {0: 0.8781600016811314, 1: 1.1610958295129339}
Epoch 1/60
980/980 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9309 - loss: 0.7773 - precision: 0.9309 - recall: 0.9309
Epoch 1: val_loss improved from None to 0.57230, saving model to outputs/checkpoints/best_dilated_se_firenet.keras

Epoch 1: finished saving model to outputs/checkpoints/best_dilated_se_firenet.keras
980/980 ━━━━━━━━━━━━━━━━━━━━ 138s 104ms/step - accuracy: 0.9770 - loss: 0.4597 - precision: 0.9770 - recall: 0.9770 - val_accuracy: 0.6405 - val_loss: 0.5723 - val_precision: 0.6405 - val_recall: 0.6405
Epoch 2/60
979/980 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.9891 - loss: 0.2609 - precision: 0.9891 - recall: 0.9891
Epoch 2: val_loss improved from 0.57230 to 0.37626, saving model to outputs/checkpoints/best_dilated_se_firenet.keras

Epoch 2: finished saving model to outputs/checkpoints/best_dilated_se_firenet.keras
980/980 ━━━━━━━━━━━━━━━━━━━━ 78s 80ms/step - accuracy: 0.9933 - loss: 0.2382 - preci

### 7. Tải xuống / Lưu trữ mô hình đã huấn luyện

Do Service Account có thể gặp giới hạn dung lượng lưu trữ (Quota 0 bytes của tài khoản dịch vụ dịch), chúng ta có 3 cách để lấy file mô hình `best_dilated_se_firenet.keras` (~1.6 MB) từ Google Colab về máy tính hoặc lưu vào Drive cá nhân của bạn:

* **Cách 1 (Khuyên dùng - Nhanh nhất):** Tải trực tiếp về máy tính qua trình duyệt.
* **Cách 2:** Gắn (mount) Google Drive cá nhân của bạn vào Colab và copy file mô hình vào đó.
* **Cách 3:** Thử tải lên qua Service Account (có thể bị lỗi Quota nếu tài khoản dịch vụ bị giới hạn).

In [ ]:
# ==========================================
# CÁCH 1: TẢI TRỰC TIẾP VỀ MÁY TÍNH QUA TRÌNH DUYỆT
# ==========================================
from google.colab import files
import os

MODEL_FILE = 'outputs/checkpoints/best_dilated_se_firenet.keras'

if os.path.exists(MODEL_FILE):
    print("Đang tải file mô hình về máy tính...")
    files.download(MODEL_FILE)
else:
    print(f"Không tìm thấy file mô hình tại {MODEL_FILE}")

In [ ]:
# ==========================================
# CÁCH 2: LƯU VÀO GOOGLE DRIVE CÁ NHÂN CỦA BẠN
# (Bỏ dấu thăng # các dòng bên dưới để chạy)
# ==========================================
# from google.colab import drive
# import shutil
# 
# drive.mount('/content/drive')
# target_drive_path = '/content/drive/MyDrive/best_dilated_se_firenet.keras'
# shutil.copy('outputs/checkpoints/best_dilated_se_firenet.keras', target_drive_path)
# print(f"Đã lưu mô hình vào Google Drive cá nhân: {target_drive_path}")

In [ ]:
# ==========================================
# CÁCH 3: TẢI LÊN DRIVE CHUNG QUA SERVICE ACCOUNT
# (Có thể báo lỗi 403 storageQuotaExceeded do giới hạn của Service Account)
# ==========================================
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

KEY_FILE = 'dilated-se-fire-10963f0a6a5a.json'
FOLDER_ID = '1b81qr2Q3Je2sN027C7MTlxonJFKystlJ'
MODEL_FILE = 'outputs/checkpoints/best_dilated_se_firenet.keras'

if os.path.exists(KEY_FILE) and os.path.exists(MODEL_FILE):
    print('Đang tải mô hình lên Google Drive sử dụng Service Account...')
    try:
        creds = service_account.Credentials.from_service_account_file(
            KEY_FILE,
            scopes=['https://www.googleapis.com/auth/drive']
        )
        drive_service = build('drive', 'v3', credentials=creds)

        file_metadata = {
            'name': 'best_dilated_se_firenet.keras',
            'parents': [FOLDER_ID]
        }
        media = MediaFileUpload(MODEL_FILE, mimetype='application/octet-stream')
        file = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        print(f'[SUCCESS] Đã lưu mô hình lên Google Drive. File ID: {file.get("id")}')
    except Exception as e:
        print(f'Lỗi khi upload mô hình lên Drive (Có thể do vượt quá Quota của Service Account): {e}')
else:
    print('Bỏ qua bước tải mô hình lên Drive (Thiếu tệp Key hoặc mô hình chưa được huấn luyện).')